In [2]:
PROJECT = "urban-access-bot"

H3_TABLE = f"{PROJECT}.bot_geo.h3_10res_london_kl"
LSOA_POSTCODE_TABLE = f"{PROJECT}.bot_geo.lsoa_postcode_map_tl"
RENTS_TABLE = f"{PROJECT}.bot_rents.monthly_rents_by_postcode_oct2024_to_sept2025_ONS_me"

OUTPUT_TABLE = f"{PROJECT}.analytics.h3_london_full_dataset"

### Версия 1

In [3]:
def generate_sql():

    sql = f"""
    CREATE OR REPLACE TABLE {OUTPUT_TABLE} AS

    WITH

    rents_h3 AS (

        SELECT
            h3.h3_id,

            AVG(SAFE_CAST(r.Mean AS FLOAT64)) AS rent_mean_avg,
            AVG(SAFE_CAST(r.Median AS FLOAT64)) AS rent_median_avg,
            AVG(SAFE_CAST(r.Lower quartile AS FLOAT64)) AS rent_lower_q_avg,
            AVG(SAFE_CAST(r.Upper quartile AS FLOAT64)) AS rent_upper_q_avg

        FROM {RENTS_TABLE} r

        JOIN {LSOA_POSTCODE_TABLE} p
            ON LOWER(TRIM(r.Postcode District)) =
               LOWER(TRIM(p.pcd7))

        JOIN {H3_TABLE} h3
            ON p.lsoa21cd = h3.LSOA code

        GROUP BY h3.h3_id
    ),

    stats_h3 AS (

        SELECT
            h.h3_id,

            AVG(d.Disabled: activities limited a lot
                + d.Disabled: activities limited a little) AS disabled,

            AVG(a.Aged 4 and under) AS preschool_age,

            AVG(a.Aged 5 to 9
                + a.Aged 10 to 14
                + a.Aged 15 to 19) AS school_age,

            AVG(a.Aged 65 to 69
                + a.Aged 70 to 74
                + a.Aged 75 to 79
                + a.Aged 80 to 84
                + a.Aged 85 and over) AS retirement_age,

            AVG(gh.Bad health
                + gh.Very bad health) AS bad_health,

            AVG(cv.value_21) AS car_van_availability,

            AVG(hh.All Households) AS households,

            AVG(imd.Index of Multiple Deprivation IMD Rank where 1 is most deprived)
                AS deprivation,

            AVG(pop.2021) AS total_population,

            AVG(emp.All usual residents aged 16 and over in employment)
                AS employment,

            AVG(q.none) AS without_education,

            AVG(q.Level 1
                + q.Level 2
                + q.Level 3) AS preuniversity_education,

            AVG(q.Apprenticeship) AS apprentice_ship,

            AVG(q.Level 4+) AS university_education

        FROM {H3_TABLE} h

        LEFT JOIN {PROJECT}.disability_by_lsoa_and_boroughs_2021_london_datastore_kl d
            ON h.LSOA code = d.LSOA code

        LEFT JOIN {PROJECT}.five_year_age_bands_by_lsoa_and_boroughs_2021_london_datastore_kl a
            ON h.LSOA code = a.LSOA code

        LEFT JOIN {PROJECT}.general_health_by_lsoa_and_boroughs_2021_london_datastore_kl gh
            ON h.LSOA code = gh.LSOA code

        LEFT JOIN {PROJECT}.household_composition_by_car_van_availability_by_lsoa_and_boroughs_2021_london_datastore_kl cv
            ON h.LSOA code = cv.LSOA code

        LEFT JOIN {PROJECT}.household_size_by_lsoa_and_boroughs_2021_london_datastore_kl hh
            ON h.LSOA code = hh.LSOA code

        LEFT JOIN {PROJECT}.index_of_multiple_deprivation_lsoa_and_boroughs_2025_gov_uk_kl imd
            ON h.LSOA code = imd.LSOA code

        LEFT JOIN {PROJECT}.mid-year_total_population_by_boroughs_1991_to_2024_ONS_kl pop
            ON h.LSOA code = pop.LSOA code

        LEFT JOIN {PROJECT}.occupation_by_lsoa_and_boroughs_2021_london_datastore_kl emp
            ON h.LSOA code = emp.LSOA code

        LEFT JOIN {PROJECT}.qualifications_by_lsoa_and_boroughs_2021_london_datastore_kl q
            ON h.LSOA code = q.LSOA code

        GROUP BY h.h3_id
    )

    SELECT
        s.*,
        r.rent_mean_avg,
        r.rent_median_avg,
        r.rent_lower_q_avg,
        r.rent_upper_q_avg

    FROM stats_h3 s
    LEFT JOIN rents_h3 r
        ON s.h3_id = r.h3_id
    """

    return sql

In [4]:
print(generate_sql())


    CREATE OR REPLACE TABLE urban-access-bot.analytics.h3_london_full_dataset AS

    WITH

    rents_h3 AS (

        SELECT
            h3.h3_id,

            AVG(SAFE_CAST(r.Mean AS FLOAT64)) AS rent_mean_avg,
            AVG(SAFE_CAST(r.Median AS FLOAT64)) AS rent_median_avg,
            AVG(SAFE_CAST(r.Lower quartile AS FLOAT64)) AS rent_lower_q_avg,
            AVG(SAFE_CAST(r.Upper quartile AS FLOAT64)) AS rent_upper_q_avg

        FROM urban-access-bot.bot_rents.monthly_rents_by_postcode_oct2024_to_sept2025_ONS_me r

        JOIN urban-access-bot.bot_geo.lsoa_postcode_map_tl p
            ON LOWER(TRIM(r.Postcode District)) =
               LOWER(TRIM(p.pcd7))

        JOIN urban-access-bot.bot_geo.h3_10res_london_kl h3
            ON p.lsoa21cd = h3.LSOA code

        GROUP BY h3.h3_id
    ),

    stats_h3 AS (

        SELECT
            h.h3_id,

            AVG(d.Disabled: activities limited a lot
                + d.Disabled: activities limited a little) AS disabled,

   

### Версия 2 (Статистика на гекс)

In [8]:
def generate_sql():

    sql = f"""
    CREATE OR REPLACE TABLE {OUTPUT_TABLE} AS

    WITH

    rents_h3 AS (

        SELECT
            h3.h3_id,

            AVG(SAFE_CAST(r.`Mean` AS FLOAT64)) AS rent_mean_avg,
            AVG(SAFE_CAST(r.`Median` AS FLOAT64)) AS rent_median_avg,
            AVG(SAFE_CAST(r.`Lower quartile` AS FLOAT64)) AS rent_lower_q_avg,
            AVG(SAFE_CAST(r.`Upper quartile` AS FLOAT64)) AS rent_upper_q_avg

        FROM {RENTS_TABLE} r

        JOIN {LSOA_POSTCODE_TABLE} p
            ON LOWER(TRIM(r.`Postcode District`)) =
               LOWER(TRIM(p.pcd7))

        JOIN {H3_TABLE} h3
            ON p.lsoa21cd = h3.`LSOA code`

        GROUP BY h3.h3_id
    ),

    lsoa_stats AS (

        SELECT
            h.h3_id,

            d.`Disabled: activities limited a lot`
                + d.`Disabled: activities limited a little`
                AS disabled_raw,

            a.`Aged 4 and under` AS preschool_age_raw,

            a.`Aged 5 to 9`
                + a.`Aged 10 to 14`
                + a.`Aged 15 to 19`
                AS school_age_raw,

            a.`Aged 65 to 69`
                + a.`Aged 70 to 74`
                + a.`Aged 75 to 79`
                + a.`Aged 80 to 84`
                + a.`Aged 85 and over`
                AS retirement_age_raw,

            gh.`Bad health`
                + gh.`Very bad health`
                AS bad_health_raw,

            cv.`value_21` AS car_van_availability_raw,

            hh.`All Households` AS households,

            imd.`Index of Multiple Deprivation IMD Rank where 1 is most deprived`
                AS deprivation,

            pop.`2021` AS total_population,

            emp.`All usual residents aged 16 and over in employment`
                AS employment_raw,

            q.`none` AS without_education_raw,

            q.`Level 1`
                + q.`Level 2`
                + q.`Level 3`
                AS preuniversity_education_raw,

            q.`Apprenticeship` AS apprentice_ship_raw,

            q.`Level 4+` AS university_education_raw

        FROM {H3_TABLE} h

        LEFT JOIN {PROJECT}.disability_by_lsoa_and_boroughs_2021_london_datastore_kl d
            ON h.`LSOA code` = d.`LSOA code`

        LEFT JOIN {PROJECT}.five_year_age_bands_by_lsoa_and_boroughs_2021_london_datastore_kl a
            ON h.`LSOA code` = a.`LSOA code`

        LEFT JOIN {PROJECT}.general_health_by_lsoa_and_boroughs_2021_london_datastore_kl gh
            ON h.`LSOA code` = gh.`LSOA code`

        LEFT JOIN {PROJECT}.household_composition_by_car_van_availability_by_lsoa_and_boroughs_2021_london_datastore_kl cv
            ON h.`LSOA code` = cv.`LSOA code`

        LEFT JOIN {PROJECT}.household_size_by_lsoa_and_boroughs_2021_london_datastore_kl hh
            ON h.`LSOA code` = hh.`LSOA code`

        LEFT JOIN {PROJECT}.index_of_multiple_deprivation_lsoa_and_boroughs_2025_gov_uk_kl imd
            ON h.`LSOA code` = imd.`LSOA code`

        LEFT JOIN {PROJECT}.mid-year_total_population_by_boroughs_1991_to_2024_ONS_kl pop
            ON h.`LSOA code` = pop.`LSOA code`

        LEFT JOIN {PROJECT}.occupation_by_lsoa_and_boroughs_2021_london_datastore_kl emp
            ON h.`LSOA code` = emp.`LSOA code`

        LEFT JOIN {PROJECT}.qualifications_by_lsoa_and_boroughs_2021_london_datastore_kl q
            ON h.`LSOA code` = q.`LSOA code`
    ),

    stats_h3 AS (

        SELECT
            h3_id,

            SUM(disabled_raw) / SUM(total_population) AS disabled,

            SUM(preschool_age_raw) / SUM(total_population) AS preschool_age,

            SUM(school_age_raw) / SUM(total_population) AS school_age,

            SUM(retirement_age_raw) / SUM(total_population) AS retirement_age,

            SUM(bad_health_raw) / SUM(total_population) AS bad_health,

            SUM(car_van_availability_raw) / SUM(households)
                AS car_van_availability,

            SUM(households) AS households,

            AVG(deprivation) AS deprivation,

            SUM(total_population) AS total_population,

            SUM(employment_raw) / SUM(total_population) AS employment,

            SUM(without_education_raw) / SUM(total_population)
                AS without_education,

            SUM(preuniversity_education_raw) / SUM(total_population)
                AS preuniversity_education,

            SUM(apprentice_ship_raw) / SUM(total_population)
                AS apprentice_ship,

            SUM(university_education_raw) / SUM(total_population)
                AS university_education

        FROM lsoa_stats
        GROUP BY h3_id
    )

    SELECT
        s.*,
        r.rent_mean_avg,
        r.rent_median_avg,
        r.rent_lower_q_avg,
        r.rent_upper_q_avg

    FROM stats_h3 s
    LEFT JOIN rents_h3 r
        ON s.h3_id = r.h3_id
    """

    return sql

In [9]:
print(generate_sql())


    CREATE OR REPLACE TABLE urban-access-bot.analytics.h3_london_full_dataset AS

    WITH

    rents_h3 AS (

        SELECT
            h3.h3_id,

            AVG(SAFE_CAST(r.`Mean` AS FLOAT64)) AS rent_mean_avg,
            AVG(SAFE_CAST(r.`Median` AS FLOAT64)) AS rent_median_avg,
            AVG(SAFE_CAST(r.`Lower quartile` AS FLOAT64)) AS rent_lower_q_avg,
            AVG(SAFE_CAST(r.`Upper quartile` AS FLOAT64)) AS rent_upper_q_avg

        FROM urban-access-bot.bot_rents.monthly_rents_by_postcode_oct2024_to_sept2025_ONS_me r

        JOIN urban-access-bot.bot_geo.lsoa_postcode_map_tl p
            ON LOWER(TRIM(r.`Postcode District`)) =
               LOWER(TRIM(p.pcd7))

        JOIN urban-access-bot.bot_geo.h3_10res_london_kl h3
            ON p.lsoa21cd = h3.`LSOA code`

        GROUP BY h3.h3_id
    ),

    lsoa_stats AS (

        SELECT
            h.h3_id,

            d.`Disabled: activities limited a lot`
                + d.`Disabled: activities limited a little`
    

### Вариант 3 (оптимизированный)

In [13]:
def generate_sql():

    sql = f"""
    CREATE OR REPLACE TABLE {OUTPUT_TABLE} AS

    WITH

    rents_h3 AS (

        SELECT
            h3.h3_id,

            AVG(SAFE_CAST(r.`Mean` AS FLOAT64)) AS rent_mean_avg,
            AVG(SAFE_CAST(r.`Median` AS FLOAT64)) AS rent_median_avg,
            AVG(SAFE_CAST(r.`Lower quartile` AS FLOAT64)) AS rent_lower_q_avg,
            AVG(SAFE_CAST(r.`Upper quartile` AS FLOAT64)) AS rent_upper_q_avg

        FROM {RENTS_TABLE} r

        JOIN {LSOA_POSTCODE_TABLE} p
            ON LOWER(TRIM(r.`Postcode District`)) =
               LOWER(TRIM(p.pcd7))

        JOIN {H3_TABLE} h3
            ON p.lsoa21cd = h3.`LSOA code`

        GROUP BY h3.h3_id
    ),

    lsoa_base AS (

        SELECT

            d.`LSOA code` AS lsoa_code,

            d.`Disabled: activities limited a lot`
            + d.`Disabled: activities limited a little`
                AS disabled_raw,

            a.`Aged 4 and under`
                AS preschool_age_raw,

            a.`Aged 5 to 9`
            + a.`Aged 10 to 14`
            + a.`Aged 15 to 19`
                AS school_age_raw,

            a.`Aged 65 to 69`
            + a.`Aged 70 to 74`
            + a.`Aged 75 to 79`
            + a.`Aged 80 to 84`
            + a.`Aged 85 and over`
                AS retirement_age_raw,

            gh.`Bad health`
            + gh.`Very bad health`
                AS bad_health_raw,

            cv.`value_21`
                AS car_van_availability_raw,

            hh.`All Households`
                AS households,

            imd.`Index of Multiple Deprivation IMD Rank where 1 is most deprived`
                AS deprivation,

            pop.`2021`
                AS total_population,

            emp.`All usual residents aged 16 and over in employment`
                AS employment_raw,

            q.`none`
                AS without_education_raw,

            q.`Level 1`
            + q.`Level 2`
            + q.`Level 3`
                AS preuniversity_education_raw,

            q.`Apprenticeship`
                AS apprentice_ship_raw,

            q.`Level 4+`
                AS university_education_raw

        FROM {PROJECT}.bot_demographics.disability_by_lsoa_and_boroughs_2021_london_datastore_kl d

        LEFT JOIN {PROJECT}.bot_demographics.five_year_age_bands_by_lsoa_and_boroughs_2021_london_datastore_kl a
            USING (`LSOA code`)

        LEFT JOIN {PROJECT}.bot_demographics.general_health_by_lsoa_and_boroughs_2021_london_datastore_kl gh
            USING (`LSOA code`)

        LEFT JOIN {PROJECT}.bot_demographics.household_composition_by_car_van_availability_by_lsoa_and_boroughs_2021_london_datastore_kl cv
            USING (`LSOA code`)

        LEFT JOIN {PROJECT}.bot_demographics.household_size_by_lsoa_and_boroughs_2021_london_datastore_kl hh
            USING (`LSOA code`)

        LEFT JOIN {PROJECT}.bot_demographics.index_of_multiple_deprivation_lsoa_and_boroughs_2025_gov_uk_kl imd
            USING (`LSOA code`)

        LEFT JOIN {PROJECT}.bot_demographics.`mid-year_total_population_by_boroughs_1991_to_2024_ONS_kl` pop
            USING (`LSOA code`)

        LEFT JOIN {PROJECT}.bot_demographics.occupation_by_lsoa_and_boroughs_2021_london_datastore_kl emp
            USING (`LSOA code`)

        LEFT JOIN {PROJECT}.bot_demographics.qualifications_by_lsoa_and_boroughs_2021_london_datastore_kl q
            USING (`LSOA code`)
    ),

    stats_h3 AS (

        SELECT
            h.h3_id,

            SUM(disabled_raw) / SUM(total_population) AS disabled_share,

            SUM(preschool_age_raw) / SUM(total_population) AS preschool_age_share,

            SUM(school_age_raw) / SUM(total_population) AS school_age_share,

            SUM(retirement_age_raw) / SUM(total_population) AS retirement_age,_share

            SUM(bad_health_raw) / SUM(total_population) AS bad_health_share,

            SUM(car_van_availability_raw) / SUM(households)
                AS car_van_availability_share,

            AVG(deprivation) AS deprivation,

            SUM(total_population) AS total_population,

            SUM(employment_raw) / SUM(total_population)
                AS employment_share,

            SUM(without_education_raw) / SUM(total_population)
                AS without_education_share,

            SUM(preuniversity_education_raw) / SUM(total_population)
                AS preuniversity_education_share,

            SUM(apprentice_ship_raw) / SUM(total_population)
                AS apprentice_ship_share,

            SUM(university_education_raw) / SUM(total_population)
                AS university_education_share

        FROM {H3_TABLE} h

        JOIN lsoa_base l
            ON h.`LSOA code` = l.lsoa_code

        GROUP BY h.h3_id
    )

    SELECT
        s.*,
        r.rent_mean_avg,
        r.rent_median_avg,
        r.rent_lower_q_avg,
        r.rent_upper_q_avg

    FROM stats_h3 s
    LEFT JOIN rents_h3 r
        ON s.h3_id = r.h3_id
    """

    return sql

print(generate_sql())


    CREATE OR REPLACE TABLE urban-access-bot.analytics.h3_london_full_dataset AS

    WITH

    rents_h3 AS (

        SELECT
            h3.h3_id,

            AVG(SAFE_CAST(r.`Mean` AS FLOAT64)) AS rent_mean_avg,
            AVG(SAFE_CAST(r.`Median` AS FLOAT64)) AS rent_median_avg,
            AVG(SAFE_CAST(r.`Lower quartile` AS FLOAT64)) AS rent_lower_q_avg,
            AVG(SAFE_CAST(r.`Upper quartile` AS FLOAT64)) AS rent_upper_q_avg

        FROM urban-access-bot.bot_rents.monthly_rents_by_postcode_oct2024_to_sept2025_ONS_me r

        JOIN urban-access-bot.bot_geo.lsoa_postcode_map_tl p
            ON LOWER(TRIM(r.`Postcode District`)) =
               LOWER(TRIM(p.pcd7))

        JOIN urban-access-bot.bot_geo.h3_10res_london_kl h3
            ON p.lsoa21cd = h3.`LSOA code`

        GROUP BY h3.h3_id
    ),

    lsoa_base AS (

        SELECT

            d.`LSOA code` AS lsoa_code,

            d.`Disabled: activities limited a lot`
            + d.`Disabled: activities limited

### Итоговый SQL скрипт

WITH

    rents_h3 AS (

        SELECT
            h3.index as h3_id,

            AVG(SAFE_CAST(r.`Mean` AS FLOAT64)) AS rent_mean_avg,
            AVG(SAFE_CAST(r.`Median` AS FLOAT64)) AS rent_median_avg,
            AVG(SAFE_CAST(r.`Lower quartile` AS FLOAT64)) AS rent_lower_q_avg,
            AVG(SAFE_CAST(r.`Upper quartile` AS FLOAT64)) AS rent_upper_q_avg

        FROM urban-access-bot.bot_rents.monthly_rents_by_postcode_oct2024_to_sept2025_ONS_me r

        JOIN urban-access-bot.bot_geo.lsoa_postcode_map_tl p
            ON LOWER(TRIM(r.`Postcode District`)) =
               LOWER(TRIM(p.pcd7))

        JOIN urban-access-bot.bot_geo.h3_10res_london_kl h3
            ON p.lsoa21cd = h3.`index`

        GROUP BY h3.index
    ),

    lsoa_base AS (

        SELECT

            d.`LSOA code` AS lsoa_code,

            d.`Disabled: activities limited a lot`
            + d.`Disabled: activities limited a little`
                AS disabled_raw,

            a.`Aged 4 and under`
                AS preschool_age_raw,

            a.`Aged 5 to 9`
            + a.`Aged 10 to 14`
            + a.`Aged 15 to 19`
                AS school_age_raw,

            a.`Aged 65 to 69`
            + a.`Aged 70 to 74`
            + a.`Aged 75 to 79`
            + a.`Aged 80 to 84`
            + a.`Aged 85 and over`
                AS retirement_age_raw,

            gh.`Bad health`
            + gh.`Very bad health`
                AS bad_health_raw,

            cv.`value_21`
                AS car_van_availability_raw,

            hh.`All Households`
                AS households,

            imd.`Index of Multiple Deprivation IMD Rank where 1 is most deprived`
                AS deprivation,

            pop.`2021`
                AS total_population,

            emp.`All usual residents aged 16 and over in employment`
                AS employment_raw,

            q.`none`
                AS without_education_raw,

            q.`Level 1`
            + q.`Level 2`
            + q.`Level 3`
                AS preuniversity_education_raw,

            q.`Apprentice-ship`
                AS apprentice_ship_raw,

            q.`Level 4+`
                AS university_education_raw

        FROM urban-access-bot.bot_demographics.disability_by_lsoa_and_boroughs_2021_london_datastore_kl d

        LEFT JOIN urban-access-bot.bot_demographics.five_year_age_bands_by_lsoa_and_boroughs_2021_london_datastore_kl a
            USING (`LSOA code`)

        LEFT JOIN urban-access-bot.bot_demographics.general_health_by_lsoa_and_boroughs_2021_london_datastore_kl gh
            USING (`LSOA code`)

        LEFT JOIN urban-access-bot.bot_demographics.household_composition_by_car_van_availability_by_lsoa_and_boroughs_2011_and_2021_london_datastore_kl cv
            on `LSOA code` = cv.lsoa21cd

        LEFT JOIN urban-access-bot.bot_demographics.household_size_by_lsoa_and_boroughs_2021_london_datastore_kl hh
            USING (`LSOA code`)

        LEFT JOIN urban-access-bot.bot_demographics.index_of_multiple_deprivation_lsoa_and_boroughs_2025_gov_uk_kl imd
            on `LSOA code` = imd.`LSOA code 2021`

        LEFT JOIN urban-access-bot.bot_demographics.`mid-year_total_population_by_boroughs_1991_to_2024_ONS_kl` pop
            on `LSOA code` = pop.`Area_code`

        LEFT JOIN urban-access-bot.bot_demographics.occupation_by_lsoa_and_boroughs_2021_london_datastore_kl emp
            USING (`LSOA code`)

        LEFT JOIN urban-access-bot.bot_demographics.qualifications_by_lsoa_and_boroughs_2021_london_datastore_kl q
            USING (`LSOA code`)
    ),

    stats_h3 AS (

        SELECT
            h.index,

            SUM(disabled_raw) / SUM(total_population) AS disabled_share,

            SUM(preschool_age_raw) / SUM(total_population) AS preschool_age_share,

            SUM(school_age_raw) / SUM(total_population) AS school_age_share,

            SUM(retirement_age_raw) / SUM(total_population) AS retirement_age_share,

            SUM(bad_health_raw) / SUM(total_population) AS bad_health_share,

            SUM(car_van_availability_raw) / SUM(households)
                AS car_van_availability_share,

            AVG(deprivation) AS deprivation,

            SUM(total_population) AS total_population,

            SUM(employment_raw) / SUM(total_population)
                AS employment_share,

            SUM(without_education_raw) / SUM(total_population)
                AS without_education_share,

            SUM(preuniversity_education_raw) / SUM(total_population)
                AS preuniversity_education_share,

            SUM(apprentice_ship_raw) / SUM(total_population)
                AS apprentice_ship_share,

            SUM(university_education_raw) / SUM(total_population)
                AS university_education_share

        FROM urban-access-bot.bot_geo.h3_10res_london_kl h

        JOIN lsoa_base l
            ON h.`index` = l.lsoa_code

        GROUP BY h.index
    )

    SELECT
        s.*,
        r.rent_mean_avg,
        r.rent_median_avg,
        r.rent_lower_q_avg,
        r.rent_upper_q_avg

    FROM stats_h3 s
    LEFT JOIN rents_h3 r
        ON s.index = r.h3_id

In [ ]:
from google.cloud import bigquery

client = bigquery.Client()

query = generate_sql()

job = client.query(query)
job.result()